# XGBoost Model — Détection de Fraude Chargeback

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix,
    precision_recall_curve
)

# ── Constantes ──────────────────────────────
TARGET        = 'target_is_fraud'
DROP_COLS     = ['customer_id']
RANDOM_STATE  = 42
N_TRIALS      = 50
TOP_N_FEATURES = 15

/Users/mpmax/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Chargement des données

In [2]:
train_df = pd.read_csv('../data/train_1.csv')
val_df   = pd.read_csv('../data/val_1.csv')
test_df  = pd.read_csv('../data/test_1.csv')

def prepare_split(df, target=TARGET, drop=DROP_COLS, include_target=True):
    cols_to_drop = [c for c in drop if c in df.columns]
    if include_target:
        cols_to_drop += [target]
    X = df.drop(columns=cols_to_drop).select_dtypes(include=[np.number])
    y = df[target] if include_target else None
    return X, y

X_train, y_train = prepare_split(train_df)
X_val,   y_val   = prepare_split(val_df)
X_test,  _       = prepare_split(test_df, include_target=False)

print(f"Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}")

Train : (233956, 21) | Val : (32000, 21) | Test : (40000, 21)


## 3. Optimisation des hyperparamètres (Optuna)

In [3]:
def objective(trial):
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators", 200, 1000),
        "max_depth"         : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"         : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight"  : trial.suggest_int("min_child_weight", 1, 10),
        "gamma"             : trial.suggest_float("gamma", 0, 5),
        "reg_alpha"         : trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda"        : trial.suggest_float("reg_lambda", 0, 5),
        "eval_metric"       : "aucpr",
        "random_state"      : RANDOM_STATE,
        "n_jobs"            : -1,
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=20)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return average_precision_score(y_val, model.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print(f"Meilleur AUC-PR : {study.best_value:.4f}")
print(f"Meilleurs params : {study.best_params}")

[I 2026-03-03 13:02:17,637] A new study created in memory with name: no-name-e3e5b4d0-7127-4899-8e51-359c62e27375
[I 2026-03-03 13:02:17,961] Trial 0 finished with value: 0.38742251608548384 and parameters: {'n_estimators': 596, 'max_depth': 4, 'learning_rate': 0.12781567968733512, 'subsample': 0.6586226549671206, 'colsample_bytree': 0.6360793904641259, 'min_child_weight': 7, 'gamma': 2.626731305462335, 'reg_alpha': 0.8649145264189584, 'reg_lambda': 0.5256308955831152}. Best is trial 0 with value: 0.38742251608548384.
[I 2026-03-03 13:02:18,407] Trial 1 finished with value: 0.38204770059474713 and parameters: {'n_estimators': 391, 'max_depth': 9, 'learning_rate': 0.045126629671727356, 'subsample': 0.5749098929933858, 'colsample_bytree': 0.7599327647592335, 'min_child_weight': 4, 'gamma': 4.310456471209113, 'reg_alpha': 1.1371703795924755, 'reg_lambda': 0.17206141106367645}. Best is trial 0 with value: 0.38742251608548384.
[I 2026-03-03 13:02:18,825] Trial 2 finished with value: 0.38573

Meilleur AUC-PR : 0.4143
Meilleurs params : {'n_estimators': 321, 'max_depth': 10, 'learning_rate': 0.09165388091795235, 'subsample': 0.7870502502606612, 'colsample_bytree': 0.6247476279325999, 'min_child_weight': 10, 'gamma': 3.9372860660849742, 'reg_alpha': 3.435966581418527, 'reg_lambda': 2.983414915768876}


## 4. Entraînement du modèle final

In [4]:
best_params = {**study.best_params, 'eval_metric': 'aucpr', 'random_state': RANDOM_STATE, 'n_jobs': -1}

model = xgb.XGBClassifier(**best_params, early_stopping_rounds=20)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print("Modèle entraîné")

Modèle entraîné


## 5. Évaluation sur le jeu de validation

In [5]:
y_proba_val = model.predict_proba(X_val)[:, 1]
y_pred_val  = model.predict(X_val)

print(f"ROC-AUC : {roc_auc_score(y_val, y_proba_val):.4f}")
print(f"AUC-PR  : {average_precision_score(y_val, y_proba_val):.4f}")
print()
print("Classification report (seuil 0.5) :")
print(classification_report(y_val, y_pred_val, digits=4))
print("Matrice de confusion :")
print(confusion_matrix(y_val, y_pred_val))

ROC-AUC : 0.8482
AUC-PR  : 0.4143

Classification report (seuil 0.5) :
              precision    recall  f1-score   support

           0     0.9325    0.9818    0.9565     29245
           1     0.5599    0.2461    0.3419      2755

    accuracy                         0.9184     32000
   macro avg     0.7462    0.6139    0.6492     32000
weighted avg     0.9005    0.9184    0.9036     32000

Matrice de confusion :
[[28712   533]
 [ 2077   678]]


## 6. Optimisation du seuil de décision

In [6]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba_val)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_idx       = f1_scores.argmax()
best_threshold = thresholds[best_idx]
best_precision = precisions[best_idx]
best_recall    = recalls[best_idx]
best_f1        = f1_scores[best_idx]

print(f"{'Seuil optimal':<15}: {best_threshold:.4f}")
print(f"{'Précision':<15}: {best_precision:.4f}")
print(f"{'Recall':<15}: {best_recall:.4f}")
print(f"{'F1-score':<15}: {best_f1:.4f}")
print()
y_pred_optimized = (y_proba_val >= best_threshold).astype(int)
print("Classification report (seuil optimisé) :")
print(classification_report(y_val, y_pred_optimized, digits=4))

Seuil optimal  : 0.2938
Précision      : 0.3879
Recall         : 0.4846
F1-score       : 0.4309

Classification report (seuil optimisé) :
              precision    recall  f1-score   support

           0     0.9503    0.9280    0.9390     29245
           1     0.3879    0.4846    0.4309      2755

    accuracy                         0.8898     32000
   macro avg     0.6691    0.7063    0.6849     32000
weighted avg     0.9019    0.8898    0.8952     32000



## 7. Importance des features

In [7]:
importance = (
    pd.Series(model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(TOP_N_FEATURES)
)
print(f"Top {TOP_N_FEATURES} features :")
print(importance)

Top 15 features :
ip_risk_z                      0.156943
is_vpn                         0.118509
num_transactions_30d           0.079297
is_new_device                  0.076768
is_new_device_x_num_devices    0.073496
is_vpn_x_ip_risk               0.068339
support_tickets_90d            0.061742
device_trust_z                 0.060012
tenure_months                  0.058765
days_since_last_login          0.056212
num_devices_30d                0.056028
failed_payments_6m             0.047087
chargebacks_12m                0.030862
age                            0.015460
annual_income_eur              0.009224
dtype: float32


## 8. Prédictions sur le jeu de test

In [ ]:
y_proba_test = model.predict_proba(X_test)[:, 1]
y_pred_test  = (y_proba_test >= best_threshold).astype(int)

submission = pd.DataFrame({
    'customer_id'    : test_df['customer_id'],
    'target_is_fraud': y_pred_test,
})
submission.to_csv('../data/predictions_test.csv', index=False)

print(f"Fraudes détectées : {y_pred_test.sum()} / {len(y_pred_test)} ({y_pred_test.mean()*100:.2f}%)")
print("Prédictions sauvegardées dans predictions_test.csv")

Fraudes détectées : 4265 / 40000 (10.66%)
✓ Prédictions sauvegardées dans predictions_test.csv
